# 🔄 Tutorial 3 (T3): Cross Validation & Learning Curves in Python
### Machine Learning for Precision Agriculture (Model Generalization & Stability)

**Goal:** Learn how Cross-Validation works and how to diagnose Overfitting/Underfitting:
1. **Why Cross-Validation?**: Why a single train-test split can give biased results.
2. **K-Fold & Stratified K-Fold Cross-Validation**: Evaluating models across 5 and 10 folds.
3. **Model Stability**: Calculating mean accuracy and standard deviation (variance across folds).
4. **Learning Curves**: Diagnosing Overfitting (High Variance) vs Underfitting (High Bias).

--- 
## ⚙️ Step 0: Import Libraries & Setup Data (For Google Colab)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 120})

# Ensure data directory exists
os.makedirs("data", exist_ok=True)
crop_csv = os.path.join("data", "Crop_recommendation.csv")

if not os.path.exists(crop_csv):
    print("Creating sample Crop_recommendation.csv for Google Colab...")
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df_c = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df_c.to_csv(crop_csv, index=False)

print("[OK] Setup complete! Data is ready.")

--- 
## 🎯 Step 1: Load Data & Preprocess

In [ ]:
df = pd.read_csv(crop_csv)

X = df.drop(columns=['label'])
y = df['label']

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Target classes count: {y.nunique()}")

--- 
## 🔄 Step 2: Implement 5-Fold and 10-Fold Stratified Cross-Validation
We evaluate model accuracy across **5 folds** and **10 folds** using `StratifiedKFold` to ensure equal class proportions in each split.

In [ ]:
model = RandomForestClassifier(n_estimators=50, random_state=42)

for k in [5, 10]:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy')
    
    print(f"\n{'='*50}")
    print(f"  {k}-FOLD STRATIFIED CROSS-VALIDATION")
    print(f"{'='*50}")
    print(f"Accuracy for each fold: {np.round(scores, 4)}")
    print(f"Mean Accuracy Score:     {scores.mean():.4f} ({scores.mean()*100:.2f}%)")
    print(f"Standard Deviation:     {scores.std():.4f}")

--- 
## 📈 Step 3: Learning Curve Generation (Overfitting vs Underfitting)
We plot **Training Score vs Validation Score** over increasing dataset size to diagnose model health.

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=model,
    X=X_scaled,
    y=y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# Plot Learning Curve
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color='royalblue', label='Training Score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='royalblue')

ax.plot(train_sizes, val_mean, 's-', color='seagreen', label='Validation Score (Cross-Val)')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='seagreen')

ax.set_title("Learning Curve: Diagnosing Overfitting vs Underfitting", fontsize=14, fontweight='bold')
ax.set_xlabel("Number of Training Samples")
ax.set_ylabel("Accuracy Score")
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

--- 
## 💡 What I Learned from Tutorial 3 (Summary for Viva / Notebook)
1. **Why Cross-Validation?**: Single train-test splits can be biased depending on random splitting. $K$-Fold cross-validation ensures every sample is tested.
2. **Stratified K-Fold**: Preserves target class ratios across all $K$ splits.
3. **Standard Deviation**: Measures model variance across folds — lower standard deviation means higher model stability.
4. **Learning Curves**: Diagnoses **Overfitting** (large gap between training and validation scores) vs **Underfitting** (low scores on both lines).